# Hybrid Network Intrusion Detection using Classical ML and Quantum ML (Qiskit)

## Objective

Build a cybersecurity intrusion detection system that compares **Classical Machine Learning** models (**Random Forest, Logistic Regression, and Support Vector Machine**) with **Quantum Machine Learning** using **Qiskit Quantum Support Vector Classifier (QSVC)**. The project demonstrates an end-to-end workflow for detecting malicious network traffic through data preprocessing, feature engineering, dimensionality reduction, and comparative model evaluation.

## Key Features

- Merge multiple network traffic datasets into a unified dataset.
- Perform data cleaning and preprocessing.
- Engineer and scale features for improved model performance.
- Select important features using **Random Forest Feature Importance**.
- Apply **Principal Component Analysis (PCA)** for dimensionality reduction.
- Train and evaluate multiple Classical Machine Learning models.
- Implement **Quantum Kernel-based QSVC** using **Qiskit**.
- Compare the performance of Classical ML and Quantum ML approaches for intrusion detection.

## Project Workflow

```text
Dataset Collection
        │
        ▼
 Data Integration
        │
        ▼
 Data Cleaning
        │
        ▼
 Feature Engineering
        │
        ▼
 Feature Scaling
        │
        ▼
 ┌──────────────────────┐
 │ Classical ML Models  │
 └──────────────────────┘
        │
        ├──────────────┐
        ▼              ▼
Feature Selection     PCA
        │              │
        └──────┬───────┘
               ▼
      Quantum Feature Map
               │
               ▼
      Quantum Kernel (Qiskit)
               │
               ▼
             QSVC
               │
               ▼
 Performance Comparison
```

## 1. Import Required Libraries
Import the libraries required for data manipulation, visualization, classical machine learning, and quantum machine learning.

In [1]:
import pandas as pd
import glob

files = glob.glob("*.csv")

print("Files found:", len(files))
print(files)

Files found: 8
['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']


## 2. Data Collection and Integration
Load multiple network traffic datasets and merge them into a unified dataset for analysis.

In [2]:
import pandas as pd
import glob

files = glob.glob("*.csv")

df_list = []

for file in files:
    print(f"Loading {file}")
    temp = pd.read_csv(file, low_memory=False)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)


Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading Monday-WorkingHours.pcap_ISCX.csv
Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading Tuesday-WorkingHours.pcap_ISCX.csv
Loading Wednesday-workingHours.pcap_ISCX.csv


## 3. Exploratory Data Analysis (EDA)
Analyze the dataset structure, class distribution, feature statistics, and identify potential data quality issues.

In [3]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
df.tail()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
2830738,53,32215,4,2,112,152,28,28,28.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830739,53,324,2,2,84,362,42,42,42.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830740,58030,82,2,1,31,6,31,0,15.5,21.92031,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830741,53,1048635,6,2,192,256,32,32,32.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830742,53,94939,4,2,188,226,47,47,47.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
df.shape

(2830743, 79)

In [6]:
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [7]:
cols = df.columns
cols = [x.strip().lower() for x in cols]
df.columns = cols

In [8]:
df.columns

Index(['destination port', 'flow duration', 'total fwd packets',
       'total backward packets', 'total length of fwd packets',
       'total length of bwd packets', 'fwd packet length max',
       'fwd packet length min', 'fwd packet length mean',
       'fwd packet length std', 'bwd packet length max',
       'bwd packet length min', 'bwd packet length mean',
       'bwd packet length std', 'flow bytes/s', 'flow packets/s',
       'flow iat mean', 'flow iat std', 'flow iat max', 'flow iat min',
       'fwd iat total', 'fwd iat mean', 'fwd iat std', 'fwd iat max',
       'fwd iat min', 'bwd iat total', 'bwd iat mean', 'bwd iat std',
       'bwd iat max', 'bwd iat min', 'fwd psh flags', 'bwd psh flags',
       'fwd urg flags', 'bwd urg flags', 'fwd header length',
       'bwd header length', 'fwd packets/s', 'bwd packets/s',
       'min packet length', 'max packet length', 'packet length mean',
       'packet length std', 'packet length variance', 'fin flag count',
       'syn flag co

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   destination port             int64  
 1   flow duration                int64  
 2   total fwd packets            int64  
 3   total backward packets       int64  
 4   total length of fwd packets  int64  
 5   total length of bwd packets  int64  
 6   fwd packet length max        int64  
 7   fwd packet length min        int64  
 8   fwd packet length mean       float64
 9   fwd packet length std        float64
 10  bwd packet length max        int64  
 11  bwd packet length min        int64  
 12  bwd packet length mean       float64
 13  bwd packet length std        float64
 14  flow bytes/s                 float64
 15  flow packets/s               float64
 16  flow iat mean                float64
 17  flow iat std                 float64
 18  flow iat max                 int64  
 19  flow iat mi

In [10]:
df.describe()

c:\Users\KIIT\anaconda3\envs\quantumml\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\KIIT\anaconda3\envs\quantumml\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,destination port,flow duration,total fwd packets,total backward packets,total length of fwd packets,total length of bwd packets,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,act_data_pkt_fwd,min_seg_size_forward,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min
count,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,...,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06
mean,8.071483e+03,1.478566e+07,9.361160e+00,1.039377e+01,5.493024e+02,1.616264e+04,2.075999e+02,1.871366e+01,5.820194e+01,6.891013e+01,...,5.418218e+00,-2.741688e+03,8.155132e+04,4.113412e+04,1.531825e+05,5.829582e+04,8.316037e+06,5.038439e+05,8.695752e+06,7.920031e+06
std,1.828363e+04,3.365374e+07,7.496728e+02,9.973883e+02,9.993589e+03,2.263088e+06,7.171848e+02,6.033935e+01,1.860912e+02,2.811871e+02,...,6.364257e+02,1.084989e+06,6.485999e+05,3.933815e+05,1.025825e+06,5.770923e+05,2.363008e+07,4.602984e+06,2.436689e+07,2.336342e+07
min,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.300000e+01,1.550000e+02,2.000000e+00,1.000000e+00,1.200000e+01,0.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.000000e+01,3.131600e+04,2.000000e+00,2.000000e+00,6.200000e+01,1.230000e+02,3.700000e+01,2.000000e+00,3.400000e+01,0.000000e+00,...,1.000000e+00,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,3.204828e+06,5.000000e+00,4.000000e+00,1.870000e+02,4.820000e+02,8.100000e+01,3.600000e+01,5.000000e+01,2.616295e+01,...,2.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.325000e+03,5.940857e+03,7.125597e+03,...,2.135570e+05,1.380000e+02,1.100000e+08,7.420000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


## 4. Data Cleaning and Preprocessing
Handle missing values, infinite values, duplicate records, and prepare the dataset for model training.

In [11]:
df.isnull().sum().sort_values(ascending=False)

flow bytes/s                   1358
flow duration                     0
destination port                  0
total backward packets            0
total length of fwd packets       0
                               ... 
idle mean                         0
idle std                          0
idle max                          0
idle min                          0
label                             0
Length: 79, dtype: int64

In [12]:
import numpy as np

inf_values = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
inf_values

np.int64(4376)

In [13]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,destination port,flow duration,total fwd packets,total backward packets,total length of fwd packets,total length of bwd packets,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,min_seg_size_forward,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label
0,54865,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830738,53,32215,4,2,112,152,28,28,28.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830739,53,324,2,2,84,362,42,42,42.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830740,58030,82,2,1,31,6,31,0,15.5,21.92031,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830741,53,1048635,6,2,192,256,32,32,32.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [14]:
inf_values_after = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
inf_values_after

np.int64(0)

In [15]:
df.isnull().sum().sort_values(ascending=False)

flow bytes/s                   2867
flow packets/s                 2867
flow duration                     0
total backward packets            0
total length of fwd packets       0
                               ... 
idle mean                         0
idle std                          0
idle max                          0
idle min                          0
label                             0
Length: 79, dtype: int64

In [16]:
df.dropna(inplace=True)

In [17]:
df.isnull().sum()

destination port               0
flow duration                  0
total fwd packets              0
total backward packets         0
total length of fwd packets    0
                              ..
idle mean                      0
idle std                       0
idle max                       0
idle min                       0
label                          0
Length: 79, dtype: int64

## 5. Feature Engineering
Generate target labels and prepare the feature matrix.

In [18]:
df['label'].value_counts()

label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [19]:
label_numeric = []
for label in df['label']:
    if label == 'BENIGN':
        label_numeric.append(0)
    else:
        label_numeric.append(1)
df['label_numeric'] = label_numeric

In [20]:
df.label_numeric.value_counts()

label_numeric
0    2271320
1     556556
Name: count, dtype: int64

In [21]:
df_sampled = df.sample(n =100000, random_state=42)

In [22]:
df_sampled.label_numeric.value_counts()

label_numeric
0    80252
1    19748
Name: count, dtype: int64

In [23]:
## unbalanced dataset to balanced dataset
benign = df[df['label_numeric'] == 0].sample(n=50_000, random_state=42)
attack = df[df['label_numeric'] == 1].sample(n=50_000, random_state=42)
df_sampled = pd.concat([benign,attack])

In [24]:
df_sampled

,destination port,flow duration,total fwd packets,total backward packets,total length of fwd packets,total length of bwd packets,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label,label_numeric
1522001,53,206,2,2,68,132,34,34,34.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
966854,443,676974,9,6,1697,1221,1349,0,188.555556,441.134931,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
491683,443,63161627,7,0,0,0,0,0,0.000000,0.000000,...,7009617.0,0.0,7009617,7009617,18700000.0,12300000.0,32100000,8023974,BENIGN,0
1676447,53,79283,1,1,56,137,56,56,56.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
1894285,443,294065,19,29,871,39643,389,0,45.842105,105.687308,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417577,444,93,1,1,0,6,0,0,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,PortScan,1
2244947,80,84951252,6,7,371,11595,353,0,61.833333,142.672235,...,1008.0,0.0,1008,1008,84800000.0,0.0,84800000,84800000,DoS Hulk,1
2396094,80,100815794,7,6,360,11595,360,0,51.428571,136.067210,...,5.0,0.0,5,5,99700000.0,0.0,99700000,99700000,DoS Hulk,1
381095,33,72,1,1,0,6,0,0,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,PortScan,1


In [25]:
df_sampled = df_sampled.sample(frac=1, random_state=42)

In [26]:
df_sampled

,destination port,flow duration,total fwd packets,total backward packets,total length of fwd packets,total length of bwd packets,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label,label_numeric
161342,80,98967500,8,5,56,11601,20,0,7.000000,5.656854,...,4510090.0,0.0,4510090,4510090,93700000.0,0.0,93700000,93700000,DDoS,1
100902,80,11988666,5,0,30,0,6,6,6.000000,0.000000,...,1999.0,0.0,1999,1999,12000000.0,0.0,12000000,12000000,DDoS,1
2499272,22,167,2,1,12,6,6,6,6.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
2261164,80,4,2,0,0,0,0,0,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,DoS Hulk,1
2327413,80,85388166,6,6,375,11595,375,0,62.500000,153.093109,...,5.0,0.0,5,5,85300000.0,0.0,85300000,85300000,DoS Hulk,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695633,53,60887,2,2,60,120,30,30,30.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
44825,80,3869973,5,0,30,0,6,6,6.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,DDoS,1
2363583,80,85647055,6,7,328,11595,310,0,54.666667,125.121807,...,10987.0,0.0,10987,10987,85500000.0,0.0,85500000,85500000,DoS Hulk,1
608591,54450,99,1,1,0,0,0,0,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0


In [27]:
x = df_sampled.drop(['label', 'label_numeric'], axis=1)

In [28]:
y = df_sampled.label_numeric

In [29]:
df.columns.where(df.dtypes == 'object')

Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
      dtype='str')

In [30]:
import numpy as np
import scipy
import sklearn

print(np.__version__)
print(scipy.__version__)
print(sklearn.__version__)

2.4.6
1.17.1
1.9.0


## 6. Feature Scaling & Train-Test-Split
Standardize numerical features using StandardScaler, divided the dataset to improve model performance.

In [31]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2, random_state=42)

In [32]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((80000, 78), (20000, 78), (80000,), (20000,))

In [33]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## 7. Classical Machine Learning Models
Train and evaluate baseline machine learning models including Logistic Regression, Random Forest, and Support Vector Machine.

In [34]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [35]:
rf.fit(x_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [36]:
from sklearn.metrics import accuracy_score
pred = rf.predict(x_test_scaled)
print(accuracy_score(y_test, pred))

0.99805


In [37]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)

In [38]:
lr.fit(x_train_scaled, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [39]:
pred = lr.predict(x_test_scaled)
print(accuracy_score(y_test, pred))

0.9304


In [40]:
from sklearn.svm import SVC
svc = SVC()

In [41]:
svc.fit(x_train_scaled, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [42]:
pred = svc.predict(x_test_scaled)
print(accuracy_score(y_test, pred))

0.95725


## 8. Feature Selection using Random Forest
Identify the most informative features based on Random Forest feature importance scores.

In [43]:
df_top20 = pd.DataFrame(rf.feature_importances_, index=x.columns, columns=['importance']).sort_values('importance', ascending=False).head(20)

In [44]:
df_top20

,importance
average packet size,0.083765
destination port,0.067275
packet length variance,0.060011
init_win_bytes_forward,0.045259
max packet length,0.043720
fwd packet length max,0.038824
avg bwd segment size,0.034909
packet length mean,0.032361
fwd packet length min,0.032112
packet length std,0.031374


In [45]:
x = df_sampled[df_top20.index]

In [46]:
x.shape

(100000, 20)

## 9. Dimensionality Reduction using Principal Component Analysis (PCA)
Reduce feature dimensionality while preserving the majority of information to improve computational efficiency.

In [47]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [48]:
from sklearn.decomposition import PCA
pca = PCA(n_components=8)
x_pca = pca.fit_transform(x_scaled)
x_pca.shape

(100000, 8)

In [49]:
print(pca.explained_variance_ratio_)
print(sum(pca.explained_variance_ratio_))

[0.43846325 0.14900867 0.10317746 0.09995951 0.05452442 0.04734165
 0.03673478 0.03130683]
0.9605165916316677


In [50]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_pca, y, test_size=0.2, random_state=42)

In [51]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((80000, 8), (20000, 8), (80000,), (20000,))

In [52]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [ ]:
rf.fit(x_train, y_train)
pred = rf.predict(x_test)
print(accuracy_score(y_test, pred)) ## Testing how much impact top 20 features rather than all features have on the accuracy of the model.

0.9946


## 10. Quantum Machine Learning with Qiskit QSVC
Construct a quantum feature map, compute the quantum kernel, and train a Quantum Support Vector Classifier (QSVC).

In [54]:
import qiskit
import qiskit_machine_learning

print(qiskit.__version__)

2.4.1


In [55]:
import pandas as pd

pca_df = pd.DataFrame(
    x_pca,
    columns=[f"PC{i+1}" for i in range(8)]
)

pca_df["label"] = y.values

In [56]:
pca_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,label
0,6.108064,-1.134370,0.436355,0.042998,0.482905,-0.255828,0.072120,-0.141581,1
1,-1.676751,-0.471117,-0.226171,-0.020555,-0.230848,-0.552325,-0.060629,-0.726583,1
2,-1.675402,-0.442396,-0.541759,-0.055238,-0.808843,0.230518,0.210939,-0.142088,0
3,-1.662482,-0.577841,-0.412550,-0.041374,-0.259477,-0.614596,-0.115357,-0.751381,1
4,5.099639,-0.201522,0.187553,0.018257,0.233007,-0.330107,-0.160617,-0.258637,1


In [57]:
df_quantum = pca_df.sample(n=100, random_state=42)

In [58]:
df_quantum

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,label
75721,-1.780822,0.282577,1.553166,0.179878,0.002480,0.016112,-0.045635,-0.153854,0
80184,-1.662482,-0.577841,-0.412569,-0.041205,-0.259477,-0.614596,-0.115357,-0.751381,1
19864,-1.676751,-0.471117,-0.226171,-0.020555,-0.230848,-0.552325,-0.060629,-0.726583,1
76699,-1.733533,0.263115,2.148406,0.249959,0.036113,0.189712,-0.565030,0.399889,0
92991,-1.730937,-0.435868,-0.564713,-0.058068,0.306288,-0.740268,0.146799,-0.101761,1
...,...,...,...,...,...,...,...,...,...
91957,-1.662489,-0.577874,-0.412031,-0.041317,-0.258622,-0.615651,-0.115807,-0.752174,1
81346,-2.155098,-0.056063,-3.297711,-0.366571,7.249265,5.320916,-1.897564,-1.449869,0
55325,-2.030121,-0.133067,-2.331611,-0.256987,4.617659,2.030954,-0.550310,-0.073909,0
29857,-0.016513,4.032714,-1.938773,-0.209826,1.166139,2.112604,-2.007636,-1.705673,0


In [59]:
df_quantum.shape

(100, 9)

In [60]:
X = df_quantum.drop("label", axis=1)
y = df_quantum["label"]

In [61]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split( X,y,test_size=0.2,random_state=42,stratify=y)

print(x_train.shape, x_test.shape)

(80, 8) (20, 8)


In [62]:
from qiskit.circuit.library import ZFeatureMap

feature_map = ZFeatureMap(
    feature_dimension=8,
    reps=1
)

feature_map.draw('text')

C:\Users\KIIT\AppData\Local\Temp\ipykernel_2564\3313760023.py:3: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._z_feature_map.ZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the z_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZFeatureMap(


┌───────────────────────────────────────────────────────┐
q_0: ┤0                                                      ├
     │                                                       │
q_1: ┤1                                                      ├
     │                                                       │
q_2: ┤2                                                      ├
     │                                                       │
q_3: ┤3                                                      ├
     │  ZFeatureMap(x[0],x[1],x[2],x[3],x[4],x[5],x[6],x[7]) │
q_4: ┤4                                                      ├
     │                                                       │
q_5: ┤5                                                      ├
     │                                                       │
q_6: ┤6                                                      ├
     │                                                       │
q_7: ┤7                                                      ├
     └───────────────────────────────────────────────────────┘

In [63]:
from qiskit_machine_learning.kernels import FidelityQuantumKernel

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

In [64]:
from qiskit_machine_learning.algorithms import QSVC

qsvc = QSVC(
    quantum_kernel=quantum_kernel
)

## 11. Performance Evaluation and Model Training Time
Compare the Training-time,accuracy_score of Classical Machine Learning models and Quantum Machine Learning.

In [65]:
import time

start = time.time()

qsvc.fit(x_train, y_train)

end = time.time()

print("Training Time:", end-start)

Training Time: 78.71538066864014


In [66]:
from sklearn.metrics import accuracy_score

pred = qsvc.predict(x_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.9


## 12.Conclusion

This project demonstrates a **hybrid cybersecurity pipeline** that combines **Classical Machine Learning** and **Quantum Machine Learning (QML)** for network intrusion detection. By integrating advanced data preprocessing, feature engineering, dimensionality reduction, and quantum-enhanced classification, the project explores the potential of QML in cybersecurity applications.

### Achievements

- Developed an end-to-end network intrusion detection workflow.
- Trained and compared multiple Classical Machine Learning models.
- Applied **Random Forest Feature Importance** for feature selection.
- Utilized **Principal Component Analysis (PCA)** for dimensionality reduction.
- Implemented a **Quantum Support Vector Classifier (QSVC)** using a quantum kernel in **Qiskit**.
- Evaluated the feasibility of Quantum Machine Learning for cybersecurity-based classification tasks.

### Future Work

- Evaluate the pipeline on larger and more diverse intrusion detection datasets.
- Execute the quantum model on real quantum hardware backends.
- Investigate hybrid quantum-classical learning architectures for improved performance and scalability.
- Explore additional Quantum Machine Learning algorithms and feature maps.
- Optimize quantum circuits to reduce computational complexity and improve execution efficiency.